In [1]:
import sys
import os

sys.path.append("..")

from utils import get_api_key, get_model_configuration, get_parameter_configuration, LLM

llm = LLM()
from tools.web_search import get_web_search_tool
web_search = get_web_search_tool()


In [2]:
print("✓ Setup valid: config, LLM, and web_search ready.")

✓ Setup valid: config, LLM, and web_search ready.


### 1. Baseline vs Tool Enhanced

In [3]:
def baseline_response(query: str) -> dict:
    system_prompt = """
    You are a healthcare assistant for Sri Lankan hospitals.
    Provide helpful information about hospital services.
    Always end with: "This is educational information; verify with the hospital / consult your clinician."
    """.strip()

    response = llm.generate(prompt=query, system_prompt=system_prompt)

    return {
        "response": response["response"],
        "tokens": response["total_tokens"],
        "latency_ms": response["latency_ms"],
        "sources": []
    }

In [4]:
def tool_enhanced_response(query: str) -> dict:
    import time
    start_time = time.time()

    # Step 1: Perform web search
    search_result = web_search.search(query)
    search_formatted = web_search.format_result(search_result)

    # Step 2: Generate response with search context
    system_prompt = """
    You are a healthcare assistant with access to current web information.
    Use the web search results to provide accurate, up-to-date operational information.
    Always cite sources and include the checked timestamp.
    Always end with: "This is educational information; verify with the hospital / consult your clinician."
    """.strip()

    prompt = f"""
    User Question: {query}
    Web Search Results:
    {search_formatted}
    Please provide a comprehensive answer using the search results. Include URLs for verification.
    """.strip()

    response = llm.generate(prompt=prompt, system_prompt=system_prompt)
    end_time = time.time()
    total_latency = int((end_time - start_time) * 1000)

    return {
        "response": response["response"],
        "tokens": response["total_tokens"],
        "search_tokens": 0,  # Tavily doesn't charge tokens
        "latency_ms": total_latency,
        "search_latency_ms": search_result["latency_ms"],
        "sources": [r["url"] for r in search_result.get("results", [])]
    }